In [42]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas import CategoricalDtype

In [43]:
chemin = "../data/raw/comptages.csv"
df = pd.read_csv(chemin, sep=";")
df.head()

,iu_ac,libelle,t_1h,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,etat_barre
0,5592,Sevres,2026-08-31T23:00:00+00:00,70.0,1.03500,Fluide,2939,Sevres-Bac,202,Sevres-Babylone,Ouvert
1,5593,Sevres,2026-08-31T23:00:00+00:00,288.0,0.54500,Fluide,202,Sevres-Babylone,2939,Sevres-Bac,Ouvert
2,4398,Quai_d'Orsay,2026-08-31T23:00:00+00:00,148.0,2.08611,Fluide,2349,Pl_de_Finlande,3522,Quai_d'Orsay-face_57-59,Ouvert
3,4500,Av_de_Versailles,2026-08-31T23:00:00+00:00,160.0,3.03222,Fluide,2398,Av_de_Versailles-Degas,2397,Pl_Clement_Ader,Ouvert
4,4644,Av_de_Versailles,2026-08-31T23:00:00+00:00,NaN,1.65778,Fluide,2470,Bd_Exelmans-Av_Versailles,2473,Av_Versailles-Boileau,Ouvert


In [44]:
df.dtypes

iu_ac                 int64
libelle              object
t_1h                 object
q                   float64
k                   float64
etat_trafic          object
iu_nd_amont           int64
libelle_nd_amont     object
iu_nd_aval            int64
libelle_nd_aval      object
etat_barre           object
dtype: object

# 1- Dates en datatime et en heure de Paris, identifiants en texte, états en category

### Dates en datatimes 

In [49]:
import pytz
from datetime import datetime
dateset = df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True, errors="coerce")
dateset

0        2026-08-31 23:00:00+00:00
1        2026-08-31 23:00:00+00:00
2        2026-08-31 23:00:00+00:00
3        2026-08-31 23:00:00+00:00
4        2026-08-31 23:00:00+00:00
                    ...           
178919   2026-03-01 00:00:00+00:00
178920   2026-03-01 00:00:00+00:00
178921   2026-03-01 00:00:00+00:00
178922   2026-03-01 00:00:00+00:00
178923   2026-03-01 00:00:00+00:00
Name: t_1h, Length: 178924, dtype: datetime64[ns, UTC]

In [51]:
df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True, errors="coerce")

In [54]:
df["t_1h"].dtype

datetime64[ns, UTC]

In [50]:
dateset.dtype

datetime64[ns, UTC]

## - Heure en heure de paris 

In [17]:
heure_paris = df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True).dt.tz_convert("Europe/Paris")

heure_paris

0        2026-09-01 01:00:00+02:00
1        2026-09-01 01:00:00+02:00
2        2026-09-01 01:00:00+02:00
3        2026-09-01 01:00:00+02:00
4        2026-09-01 01:00:00+02:00
                    ...           
178919   2026-03-01 01:00:00+01:00
178920   2026-03-01 01:00:00+01:00
178921   2026-03-01 01:00:00+01:00
178922   2026-03-01 01:00:00+01:00
178923   2026-03-01 01:00:00+01:00
Name: t_1h, Length: 178924, dtype: datetime64[ns, Europe/Paris]

In [18]:
heure_paris.dtype

datetime64[ns, Europe/Paris]

##  Identifiants en texte (pas en nombres)

In [19]:
for col in ["iu_ac", "iu_nd_amont", "iu_nd_aval"]:
    df[col] = df[col].astype("string")

## Libellés en texte

In [20]:
for col in ["libelle", "libelle_nd_amont", "libelle_nd_aval"]:
    df[col] = df[col].astype("string")


## # États en catégories

In [21]:
categorie_trafic = df["etat_trafic"] = pd.Categorical(
    df["etat_trafic"],
    categories=["Inconnu", "Fluide", "Pré-saturé", "Saturé", "Bloqué"],
    ordered=True,
)
barre_categorie = df["etat_barre"] = pd.Categorical(
    df["etat_barre"],
    categories=["Inconnu", "Ouvert", "Barré", "Invalide"],
)

In [22]:
categorie_trafic

['Fluide', 'Fluide', 'Fluide', 'Fluide', 'Fluide', ..., 'Fluide', 'Fluide', 'Fluide', 'Fluide', 'Fluide']
Length: 178924
Categories (5, object): ['Inconnu' < 'Fluide' < 'Pré-saturé' < 'Saturé' < 'Bloqué']

In [23]:
barre_categorie

['Ouvert', 'Ouvert', 'Ouvert', 'Ouvert', 'Ouvert', ..., 'Ouvert', 'Ouvert', 'Ouvert', 'Ouvert', 'Ouvert']
Length: 178924
Categories (4, object): ['Inconnu', 'Ouvert', 'Barré', 'Invalide']

## # Mesures numériques

In [24]:
q_num = df["q"] = pd.to_numeric(df["q"], errors="coerce")
k_num = df["k"] = pd.to_numeric(df["k"], errors="coerce")

In [25]:
q_num
k_num

0         1.03500
1         0.54500
2         2.08611
3         3.03222
4         1.65778
           ...   
178919    3.63056
178920    3.63056
178921    2.25000
178922    1.36056
178923    4.16833
Name: k, Length: 178924, dtype: float64

# ---

## Étape 2 — Doublons

La clé naturelle d'une mesure est le couple **arc + heure** : (`iu_ac`, `t_1h`). Une même heure sur un même tronçon ne doit apparaître qu'une fois.

In [26]:
cle = ["iu_ac", "t_1h"]

n_doublons = int(df.duplicated(subset=cle).sum())
n_lignes_impliquees = int(df.duplicated(subset=cle, keep=False).sum())
print(f"Clé testée : {cle}")
n_doublons
n_lignes_impliquees

Clé testée : ['iu_ac', 't_1h']


0

In [27]:
if n_doublons > 0:
    print("\nExemples :")
    print(df.loc[df.duplicated(subset=cle, keep=False)].sort_values(cle).head(6))

Résultat du test :
- lignes en doublon (au-delà de la première) : **0**
- lignes impliquées dans au moins un doublon : **0**

Donc chaque couple « tronçon × heure » est unique. On ne retire aucune ligne pour cause de doublon.


In [28]:
df.nunique()

iu_ac                  41
libelle                 3
t_1h                 4364
q                    1361
k                   27503
etat_trafic             5
iu_nd_amont            24
libelle_nd_amont       24
iu_nd_aval             26
libelle_nd_aval        26
etat_barre              3
dtype: int64

---

## Étape 3 — Colonnes supprimées


On vérifie d'abord si les libellés de nœuds apportent quelque chose de plus que leurs identifiants.

In [29]:
# Test : un identifiant de nœud correspond-il toujours au même libellé ?
amont = df.groupby("iu_nd_amont", observed=True)["libelle_nd_amont"].nunique()
aval = df.groupby("iu_nd_aval", observed=True)["libelle_nd_aval"].nunique()

print("Nombre max de libellés distincts pour un même iu_nd_amont :", int(amont.max()))
print("Nombre max de libellés distincts pour un même iu_nd_aval  :", int(aval.max()))
print()
print("Si ces deux nombres valent 1, le libellé est redondant avec l'identifiant.")

Nombre max de libellés distincts pour un même iu_nd_amont : 1
Nombre max de libellés distincts pour un même iu_nd_aval  : 1

Si ces deux nombres valent 1, le libellé est redondant avec l'identifiant.


In [30]:
colonnes_avant = list(df.columns)
a_supprimer = ["libelle_nd_amont", "libelle_nd_aval"]

df = df.drop(columns=a_supprimer)

print("Colonnes avant :", colonnes_avant)
print("Colonnes enlevées :", a_supprimer)
print("Colonnes après :", list(df.columns))

Colonnes avant : ['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'iu_nd_amont', 'libelle_nd_amont', 'iu_nd_aval', 'libelle_nd_aval', 'etat_barre']
Colonnes enlevées : ['libelle_nd_amont', 'libelle_nd_aval']
Colonnes après : ['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'iu_nd_amont', 'iu_nd_aval', 'etat_barre']


### Étape 4.1 — Que veulent dire « Barré » et « Invalide » sur nos axes ?

D'abord on va vérfier les effectifs

In [31]:
print("Effectifs etat_barre :")
effectif_etat_barre = df["etat_barre"].value_counts(dropna=False)
effectif_etat_barre

Effectifs etat_barre :


etat_barre
Ouvert      164545
Invalide     14183
Barré          196
Inconnu          0
Name: count, dtype: int64

On évalue le lien avec les trous de `q` et `k`.

In [32]:
resume = (
    df.groupby("etat_barre", observed=True)
    .agg(
        n=("iu_ac", "size"),
        part_q_manquant=("q", lambda s: s.isna().mean()),
        part_k_manquant=("k", lambda s: s.isna().mean()),
    )
)
print(resume.round(3))

                 n  part_q_manquant  part_k_manquant
etat_barre                                          
Ouvert      164545            0.052            0.103
Barré          196            0.092            0.133
Invalide     14183            1.000            1.000


In [33]:
print("Répartition Barré / Invalide par axe :")
print(
    pd.crosstab(df["libelle"], df["etat_barre"], normalize="index")
    .reindex(columns=["Ouvert", "Barré", "Invalide", "Inconnu"])
    .round(3)
)

Répartition Barré / Invalide par axe :
etat_barre        Ouvert  Barré  Invalide  Inconnu
libelle                                           
Av_de_Versailles   0.957  0.000     0.043      NaN
Quai_d'Orsay       0.923  0.003     0.074      NaN
Sevres             0.856  0.000     0.144      NaN


on retient que Barré = rue coupée (peu d'heures) ; Invalide = capteur / donnée inutilisable (beaucoup d'heures, et aucun débit ni occupation).

### La longueur des trous ?

On regarde, **par arc**, les suites d'heures consécutives où `q` ou `k` manque.

In [34]:
# Generé par l'IA

def longueurs_trous(manquant):
    """Liste des longueurs de runs True consécutifs."""
    longueurs = []
    debut = None
    for i, est_manquant in enumerate(manquant):
        if est_manquant and debut is None:
            debut = i
        elif (not est_manquant) and debut is not None:
            longueurs.append(i - debut)
            debut = None
    if debut is not None:
        longueurs.append(len(manquant) - debut)
    return longueurs


def bilan_trous(colonne):
    trous = []
    for _, groupe in df.sort_values(["iu_ac", "t_1h"]).groupby("iu_ac", sort=False):
        trous.extend(longueurs_trous(groupe[colonne].isna().to_list()))
    s = pd.Series(trous, dtype="int64")
    if s.empty:
        return {"n_trous": 0}
    return {
        "n_trous": int(len(s)),
        "longueur_mediane": float(s.median()),
        "longueur_p90": float(s.quantile(0.9)),
        "longueur_max": int(s.max()),
        "part_trous_1h": float((s == 1).mean()),
        "part_trous_<=3h": float((s <= 3).mean()),
        "part_trous_<=6h": float((s <= 6).mean()),
        "part_trous_>24h": float((s > 24).mean()),
    }


bilan_q = bilan_trous("q")
bilan_k = bilan_trous("k")

print("Trous sur q (débit) :")
for k, v in bilan_q.items():
    print(f"  {k}: {v}")
print()
print("Trous sur k (occupation) :")
for k, v in bilan_k.items():
    print(f"  {k}: {v}")

Trous sur q (débit) :
  n_trous: 5069
  longueur_mediane: 1.0
  longueur_p90: 3.0
  longueur_max: 4364
  part_trous_1h: 0.6780430065101598
  part_trous_<=3h: 0.9204971394752417
  part_trous_<=6h: 0.9713947524166502
  part_trous_>24h: 0.008680213059775104

Trous sur k (occupation) :
  n_trous: 4051
  longueur_mediane: 1.0
  longueur_p90: 3.0
  longueur_max: 4364
  part_trous_1h: 0.7035299925944212
  part_trous_<=3h: 0.9222414218711429
  part_trous_<=6h: 0.9688965687484572
  part_trous_>24h: 0.007652431498395458


Interpretation : 
le trou **typique** dure 1 heure, et plus de 9 trous sur 10 durent au plus 3 heures. Il reste toutefois quelques coupures très longues (jusqu'à des mois sur un arc) : ce ne sont pas des trous à interpoler.

### Quelle méthode de nettoyage choisir 

In [35]:
n = len(df)
part_invalide = (df["etat_barre"] == "Invalide").mean()
part_barre = (df["etat_barre"] == "Barré").mean()
part_q_na = df["q"].isna().mean()
part_k_na = df["k"].isna().mean()

# Parmi les heures encore « Ouvert », reste-t-il des trous ?
ouvert = df["etat_barre"] == "Ouvert"
part_q_na_ouvert = df.loc[ouvert, "q"].isna().mean()
part_k_na_ouvert = df.loc[ouvert, "k"].isna().mean()

print(f"Lignes totales                 : {n}")
print(f"Part Barré                     : {100 * part_barre:.2f} %")
print(f"Part Invalide                  : {100 * part_invalide:.2f} %")
print(f"Part q manquant (tout)         : {100 * part_q_na:.2f} %")
print(f"Part k manquant (tout)         : {100 * part_k_na:.2f} %")
print(f"Part q manquant si Ouvert      : {100 * part_q_na_ouvert:.2f} %")
print(f"Part k manquant si Ouvert      : {100 * part_k_na_ouvert:.2f} %")

Lignes totales                 : 178924
Part Barré                     : 0.11 %
Part Invalide                  : 7.93 %
Part q manquant (tout)         : 12.76 %
Part k manquant (tout)         : 17.42 %
Part q manquant si Ouvert      : 5.24 %
Part k manquant si Ouvert      : 10.31 %


In [36]:
# 1) On retire Barré et Invalide : ce ne sont pas des heures de trafic mesurable.
avant = len(df)
df = df.loc[df["etat_barre"] == "Ouvert"].copy()
print(f"Après filtre Ouvert : {len(df)} lignes (retirées : {avant - len(df)})")

# 2) Sur les trous restants, interpolation horaire limitée dans le temps, par arc.
#    Limite = 3 heures : au-delà, on ne invente pas une longue coupure.
df = df.sort_values(["iu_ac", "t_1h"])

def imputer_par_arc(groupe):
    g = groupe.set_index("t_1h").sort_index()
    g["q"] = g["q"].interpolate(method="time", limit=3)
    g["k"] = g["k"].interpolate(method="time", limit=3)
    return g.reset_index()


df = df.groupby("iu_ac", group_keys=False).apply(imputer_par_arc)

print(f"Part q manquant après impute : {100 * df['q'].isna().mean():.2f} %")
print(f"Part k manquant après impute : {100 * df['k'].isna().mean():.2f} %")

# 3) Les trous trop longs restants : on enlève la ligne (pas d'invention).
avant2 = len(df)
df = df.dropna(subset=["q", "k"]).copy()
print(f"Après dropna q/k : {len(df)} lignes (retirées : {avant2 - len(df)})")

Après filtre Ouvert : 164545 lignes (retirées : 14379)


C:\Users\zBalde\AppData\Local\Temp\ipykernel_6440\1299167612.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("iu_ac", group_keys=False).apply(imputer_par_arc)


Part q manquant après impute : 3.88 %
Part k manquant après impute : 9.77 %
Après dropna q/k : 142087 lignes (retirées : 22458)


---

## Étape 5 — Export des données propres

In [37]:


dossier_sortie = Path("data/processed")
dossier_sortie.mkdir(parents=True, exist_ok=True)
df.to_csv(dossier_sortie / "comptages_propre.csv", index=False)
print(f"{len(df)} lignes enregistrées dans {dossier_sortie / 'comptages_propre.csv'}")

142087 lignes enregistrées dans data\processed\comptages_propre.csv


In [38]:
print(Path.cwd())

C:\Users\zBalde\Desktop\Aivancity_M2\analyse_et_transfo_officiel\notebooks
